# UniProt — API & bulk

`biodb.uniprot` wraps both REST (per-protein) and FTP (whole-corpus
FASTA). Requires the `[protein]` extra for FASTA parsing (Biopython).

| Mode | Functions | What it returns |
|---|---|---|
| **REST API** | `query_protein`, `get_sequences`, `get_features`, `get_dbxrefs` | Parsed UniProt records — features, sequences, cross-references |
| **Bulk FTP** | `download_swissprot_fasta`, `download_trembl_fasta`, `iter_fasta_records`, `count_swissprot_records` | The canonical Swiss-Prot / TrEMBL FASTA distributions |

## 1. REST API — one accession at a time

`query_protein` returns Biopython `SeqRecord` objects (a list, so it's
safely reusable — unlike the upstream `SeqIO.parse` iterator).

In [1]:
from biodb import uniprot

records = uniprot.query_protein("P12345")
records[0]

SeqRecord(seq=Seq('MALLHSARVLSGVASAFHPGLAAAASARASSWWAHVEMGPPDPILGVTEAYKRD...VTK'), id='P12345', name='AATM_RABIT', description='Aspartate aminotransferase, mitochondrial', dbxrefs=['AlphaFoldDB:P12345', 'CDD:cd00609', 'CTD:2806', 'DOI:10.1093/oxfordjournals.jbchem.a135186', 'EC:2.6.1.1', 'EC:2.6.1.7', 'EMBL:AAGW02052878', 'FunCoup:P12345', 'FunFam:3.40.640.10:FF:000026', 'FunFam:3.90.1150.10:FF:000001', 'FunFam:3.90.1150.10:FF:000160', 'GO:GO:0004069', 'GO:GO:0005739', 'GO:GO:0005759', 'GO:GO:0005886', 'GO:GO:0006103', 'GO:GO:0006457', 'GO:GO:0006531', 'GO:GO:0006533', 'GO:GO:0006536', 'GO:GO:0006869', 'GO:GO:0016212', 'GO:GO:0030170', 'Gene3D:3.40.640.10', 'Gene3D:3.90.1150.10', 'GeneID:100348732', 'HOGENOM:CLU_032440_1_2_1', 'InParanoid:P12345', 'InterPro:IPR000796', 'InterPro:IPR004838', 'InterPro:IPR004839', 'InterPro:IPR015421', 'InterPro:IPR015422', 'InterPro:IPR015424', 'KEGG:ocu:100348732', 'NCBI Taxonomy:9986', 'NCBIfam:NF006719', 'OMA:VGACTIV', 'OrthoDB:6752799at2759', 'PANTH

`get_features` extracts the feature table from a UniProt record as a
DataFrame.

In [2]:
features = uniprot.get_features("P38398")  # BRCA1
features.head()

,id,type,start,end,description,evidence,original,variation,ref,length
0,PRO_0000055830,chain,0,1863,Breast cancer type 1 susceptibility protein,NaN,NaN,NaN,NaN,1863
1,<unknown id>,domain,1641,1736,BRCT 1,2,NaN,NaN,NaN,95
2,<unknown id>,domain,1755,1855,BRCT 2,2,NaN,NaN,NaN,100
3,<unknown id>,zinc finger region,23,65,RING-type,3,NaN,NaN,NaN,42
4,<unknown id>,region of interest,229,270,Disordered,4,NaN,NaN,NaN,41


`get_dbxrefs` pulls every cross-reference (Ensembl, RefSeq, HGNC,
PDB, …) — useful for downstream ID mapping.

In [3]:
xrefs = uniprot.get_dbxrefs("P38398")
print(xrefs.shape, list(xrefs.columns))
xrefs["db"].value_counts().head(10)

(594, 3) ['dbxref', 'db', 'id']


db
PubMed      106
DOI         103
GO           77
RefSeq       33
PDB          32
PDBsum       32
Reactome     27
EMBL         16
Ensembl      13
InterPro      9
Name: count, dtype: int64

## 2. Bulk FTP FASTA

Swiss-Prot is the manually-reviewed slice (~90 MB gzipped, ~570 k
entries). TrEMBL is the auto-annotated remainder — much larger
(**~50 GB gzipped**).

Both helpers stream the file and cache it under
`~/.cache/biodb/uniprot/`.

Verify the FTP URL is alive without pulling 90 MB:

In [4]:
import requests

url = f"{uniprot.UNIPROT_FTP_BASE_URL}/{uniprot.SWISSPROT_FASTA_FILENAME}"
head = requests.head(url, timeout=10, allow_redirects=True)
print(f"{head.status_code} {head.headers.get('content-length')} bytes")

200 93457057 bytes


Once downloaded, stream-iterate without loading the whole file into
RAM (essential for TrEMBL):

```python
n = 0
for rec in uniprot.iter_fasta_records(swissprot=True):
    n += 1
    if n >= 5:
        print(rec.id, len(rec.seq))
        break
```

Counting all records is just one streaming pass:

```python
total = uniprot.count_swissprot_records()  # ~570 k
```